[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_07_cifar10_cnn/task_7_cifar10_cnn.ipynb)

# Week 7 · CNN from scratch: CIFAR-10

Classify 32 x 32 colour photographs of 10 everyday classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck) with a convolutional neural network trained from scratch. Metric: accuracy; target at least 80 %.

Working in Colab? Replace `fiit-ba` in the badge URL with your GitHub username to open the copy in your fork, and run the setup cell below. **Use a GPU this week** (Colab: *Runtime -> Change runtime type -> T4 GPU*); on a laptop CPU develop with `SUBSET` set and move to a GPU for the real runs.

In [ ]:
# Colab setup (does nothing when you run locally)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb

# --- Configuration ---
SEED = 42
SUBSET = None        # e.g. 256 while debugging: train on only that many rows (see labs/README.md); None = everything
DATA_DIR = Path(os.environ.get("ZNEUS_DATA_DIR", "data"))

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## Data

Download `train.npz` and `test.npz` from the *Data* tab of the Kaggle competition (link given at the lab) into `data/kaggle/` next to this notebook (or into `$ZNEUS_DATA_DIR/kaggle/`). With the Kaggle CLI: `uv run kaggle competitions download -c <competition-slug> -p data/kaggle` and unzip there (about 170 MB).

| file | content |
|---|---|
| `train.npz` | `x`: 50 000 images, `uint8` array of shape `(50000, 32, 32, 3)` (height, width, RGB), pixel values 0..255; `y`: labels 0..9, shape `(50000,)` |
| `test.npz` | `x`: 10 000 test images `(10000, 32, 32, 3)`, **no labels**, shuffled order; row `i` is the id `i` of your submission |
| `sample_submission.csv` | the required format: `id,label` |

Labels: 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer, 5 dog, 6 frog, 7 horse, 8 ship, 9 truck. The images are stored channels-last.

The cell below loads the files into `x_all`, `y_all` (all labelled images) and `x_test`. Without the Kaggle files it falls back to the torchvision copy of CIFAR-10 (170 MB download), so the notebook runs anywhere; the fallback test images are in a different order, so that `submission.csv` will not score on Kaggle.

In [ ]:
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
KAGGLE_DIR = DATA_DIR / "kaggle"

if (KAGGLE_DIR / "train.npz").exists() and (KAGGLE_DIR / "test.npz").exists():
    train = np.load(KAGGLE_DIR / "train.npz")
    x_all, y_all = train["x"], train["y"]
    x_test = np.load(KAGGLE_DIR / "test.npz")["x"]
    y_test_local = None
    print("Kaggle files found in", KAGGLE_DIR)
else:
    from torchvision import datasets

    tr = datasets.CIFAR10(DATA_DIR / "cifar10", train=True, download=True)
    te = datasets.CIFAR10(DATA_DIR / "cifar10", train=False, download=True)
    x_all, y_all = tr.data, np.asarray(tr.targets, dtype=np.int64)
    x_test, y_test_local = te.data, np.asarray(te.targets, dtype=np.int64)
    print(f"No Kaggle files in {KAGGLE_DIR} -> torchvision CIFAR-10 (its test ids are NOT the Kaggle ids)")

if SUBSET is not None:
    x_all, y_all = x_all[:SUBSET], y_all[:SUBSET]

print(f"x_all {x_all.shape} {x_all.dtype}   y_all {y_all.shape} {y_all.dtype}   x_test {x_test.shape} {x_test.dtype}")

## Your task

Train the best CNN you can from scratch (no pretrained weights), log every run to Weights & Biases (project `zneus-2026`, run names `week07-...`) and submit your test predictions to Kaggle; the target is at least 80 % on the leaderboard. Your code goes into the cell below; it must end with `test_pred` (a numpy array, a list or a torch tensor), one predicted class (an integer 0..9) per row of `x_test`, in the same order. The last cell writes `submission.csv`.

`wandb login` once in a terminal (API key from https://wandb.ai/authorize); without an account set `WANDB_MODE=offline` and `wandb sync` the runs later. Set the project `zneus-2026` to **Public** before you hand in.

In [ ]:
# TODO: your solution. When this cell has run, `test_pred` must hold one predicted class per row of x_test.
test_pred = ...

## Kaggle submission

`submission.csv` needs exactly the columns `id,label`, 10 000 rows (`id` = row index in `test.npz`, `label` an integer 0..9), no index column. Upload it on the Kaggle competition page (*Submit Predictions*) or with `uv run kaggle competitions submit -c <competition-slug> -f submission.csv -m "week 7"`.

In [ ]:
assert test_pred is not ..., "fill in the solution cell above: test_pred is still `...`"
if torch.is_tensor(test_pred):
    test_pred = test_pred.detach().cpu().numpy()
test_pred = np.asarray(test_pred).reshape(-1).astype(int)
assert len(test_pred) == len(x_test), f"test_pred has {len(test_pred)} values, expected one per test image ({len(x_test)})"
assert ((test_pred >= 0) & (test_pred <= 9)).all(), "labels must be integers 0..9"

submission = pd.DataFrame({"id": np.arange(len(x_test)), "label": test_pred})
submission.to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(submission)} rows, columns {list(submission.columns)}")
if y_test_local is not None:
    local_accuracy = (test_pred == y_test_local).mean()
    print(f"accuracy on the torchvision test split: {local_accuracy:.4f}")
submission.head()

## Before you hand in

- [ ] the notebook runs top to bottom and is committed and pushed to your fork (`data/`, `wandb/` and `submission.csv` are git-ignored, leave them out),
- [ ] `submission.csv` is on the Kaggle competition leaderboard under your AIS login,
- [ ] your W&B project `zneus-2026` is public and contains your runs; hand in the link.